# Análise de Consumo de Eletrodomésticos — EX01

Base: `EX01_AMOSTRA.csv` (consumo elétrico de eletrodomésticos e variáveis ambientais de temperatura e umidade).

Roteiro: carregamento e exploração inicial, renomeação de colunas, identificação do consumo máximo, filtro por limiar de 70%, contagem e percentual, filtro combinado com temperatura e comparação final.

> Mantenha o arquivo `EX01_AMOSTRA.csv` na mesma pasta deste notebook.

## 1. Carregamento da amostra

Importação do pandas e leitura do CSV. `head()` mostra as 5 primeiras linhas para conferir se as colunas foram lidas corretamente.

In [1]:
import pandas as pd

df = pd.read_csv("EX01_AMOSTRA.csv")

df.head()

,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3
0,40,0,20.8900,35.4000,17.7600,39.1633,20.2900,36.9000
1,90,10,21.8900,53.1000,21.2900,45.3600,21.6333,49.2267
2,50,0,21.3900,35.5000,17.6333,40.5300,21.6667,35.2000
3,50,0,21.3900,41.0333,23.8900,34.8400,22.0333,36.9333
4,70,0,19.9633,35.1267,16.4633,40.1267,20.0000,36.4000


`shape` retorna a dimensão da amostra no formato (linhas, colunas).

In [2]:
print("Dimensão (linhas, colunas):", df.shape)
print("Total de registros:", df.shape[0])
print("Total de atributos:", df.shape[1])

Dimensão (linhas, colunas): (1974, 8)
Total de registros: 1974
Total de atributos: 8


`info()` mostra os tipos de dados de cada coluna e a quantidade de valores não nulos, útil para verificar se há dados faltantes.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1974 entries, 0 to 1973
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Appliances  1974 non-null   int64  
 1   lights      1974 non-null   int64  
 2   T1          1974 non-null   float64
 3   RH_1        1974 non-null   float64
 4   T2          1974 non-null   float64
 5   RH_2        1974 non-null   float64
 6   T3          1974 non-null   float64
 7   RH_3        1974 non-null   float64
dtypes: float64(6), int64(2)
memory usage: 123.5 KB


`describe()` traz as estatísticas descritivas (contagem, média, desvio padrão, mínimo, quartis e máximo) de todas as colunas numéricas.

In [4]:
df.describe()

,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3
count,1974.000000,1974.000000,1974.000000,1974.000000,1974.000000,1974.000000,1974.000000,1974.000000
mean,93.044580,3.839919,21.664703,40.219193,20.293635,40.411386,22.231615,39.209733
std,93.826149,7.937076,1.574883,3.971923,2.126958,4.037686,1.957275,3.288818
min,20.000000,0.000000,16.790000,27.733300,16.100000,24.063300,17.200000,30.133300
25%,50.000000,0.000000,20.700000,37.205825,18.790000,37.760000,20.700000,36.900000
50%,60.000000,0.000000,21.600000,39.590000,20.000000,40.433300,22.100000,38.433300
75%,100.000000,0.000000,22.600000,42.991675,21.500000,43.228325,23.290000,41.741250
max,770.000000,50.000000,26.260000,57.496700,28.917500,56.026700,29.198600,49.226700


## 2. Renomeação dos atributos

`Appliances` passa a ser `Consumo_Eletrodomesticos` e os atributos ambientais recebem nomes autoexplicativos.

Referência do dataset original: `T1`/`RH_1` = cozinha, `T2`/`RH_2` = sala de estar, `T3`/`RH_3` = área de serviço. `lights` é o consumo do sistema de iluminação.

In [5]:
novos_nomes = {
    "Appliances": "Consumo_Eletrodomesticos",  # consumo em Wh
    "lights": "Consumo_Luzes",                 # consumo em Wh
    "T1": "Temp_Cozinha",                      # atributo ambiental 1
    "RH_1": "Umid_Cozinha",                    # atributo ambiental 2
    "T2": "Temp_Sala",                         # atributo ambiental 3
    "RH_2": "Umid_Sala",                       # atributo ambiental 4
    "T3": "Temp_AreaServico",                  # atributo ambiental 5
    "RH_3": "Umid_AreaServico",                # atributo ambiental 6
}

df = df.rename(columns=novos_nomes)

print("Colunas após a renomeação:")
for coluna in df.columns:
    print("-", coluna)

Colunas após a renomeação:
- Consumo_Eletrodomesticos
- Consumo_Luzes
- Temp_Cozinha
- Umid_Cozinha
- Temp_Sala
- Umid_Sala
- Temp_AreaServico
- Umid_AreaServico


In [6]:
df.head()

,Consumo_Eletrodomesticos,Consumo_Luzes,Temp_Cozinha,Umid_Cozinha,Temp_Sala,Umid_Sala,Temp_AreaServico,Umid_AreaServico
0,40,0,20.8900,35.4000,17.7600,39.1633,20.2900,36.9000
1,90,10,21.8900,53.1000,21.2900,45.3600,21.6333,49.2267
2,50,0,21.3900,35.5000,17.6333,40.5300,21.6667,35.2000
3,50,0,21.3900,41.0333,23.8900,34.8400,22.0333,36.9333
4,70,0,19.9633,35.1267,16.4633,40.1267,20.0000,36.4000


## 3. Maior consumo registrado

Valor máximo da coluna de consumo de eletrodomésticos na amostra.

In [7]:
consumo_maximo = df["Consumo_Eletrodomesticos"].max()

print("Maior consumo registrado:", consumo_maximo, "Wh")

Maior consumo registrado: 770 Wh


## 4. Limiar de 70% do máximo e DataFrame filtrado

O limiar é 70% do consumo máximo. Em seguida é criado um DataFrame apenas com os registros que ficaram **acima** desse valor.

In [8]:
limiar_70 = 0.7 * consumo_maximo

df_alto_consumo = df[df["Consumo_Eletrodomesticos"] > limiar_70]

print("Limiar (70% do máximo):", limiar_70, "Wh")
print("Dimensão do DataFrame filtrado:", df_alto_consumo.shape)

df_alto_consumo.head()

Limiar (70% do máximo): 539.0 Wh
Dimensão do DataFrame filtrado: (22, 8)


,Consumo_Eletrodomesticos,Consumo_Luzes,Temp_Cozinha,Umid_Cozinha,Temp_Sala,Umid_Sala,Temp_AreaServico,Umid_AreaServico
352,620,0,24.3900,44.3333,25.3700,37.7360,26.7300,38.8633
427,650,0,20.8900,36.7000,18.2900,39.6633,21.2000,36.0000
453,560,30,21.3567,36.0900,19.5667,37.1633,21.6333,35.0300
616,570,10,24.8900,32.6667,23.5600,31.1400,23.6000,30.8233
696,590,30,19.8567,47.6633,18.7000,34.1000,18.4267,37.8333


## 5. Quantidade e percentual dos registros selecionados

Contagem absoluta dos registros acima do limiar e quanto isso representa do total da amostra.

In [9]:
total_registros = len(df)
qtd_alto_consumo = len(df_alto_consumo)
percentual_alto_consumo = (qtd_alto_consumo / total_registros) * 100

print("Registros acima do limiar:", qtd_alto_consumo)
print("Total da amostra:", total_registros)
print(f"Percentual da amostra: {percentual_alto_consumo:.2f}%")

Registros acima do limiar: 22
Total da amostra: 1974
Percentual da amostra: 1.11%


## 6. Temperatura média e filtro com dois critérios

Cálculo da média da temperatura da cozinha (antiga coluna `T1`) e criação de um segundo DataFrame que exige, ao mesmo tempo, consumo acima do limiar de 70% **e** temperatura acima da média. O operador `&` combina as duas condições.

In [10]:
temp_media = df["Temp_Cozinha"].mean()

print(f"Temperatura média (Temp_Cozinha / T1): {temp_media:.2f} °C")

Temperatura média (Temp_Cozinha / T1): 21.66 °C


In [11]:
df_duplo_criterio = df[
    (df["Consumo_Eletrodomesticos"] > limiar_70) & (df["Temp_Cozinha"] > temp_media)
]

qtd_duplo_criterio = len(df_duplo_criterio)
percentual_duplo_criterio = (qtd_duplo_criterio / total_registros) * 100

print("Registros com consumo alto E temperatura acima da média:", qtd_duplo_criterio)
print(f"Percentual da amostra: {percentual_duplo_criterio:.2f}%")

df_duplo_criterio.head()

Registros com consumo alto E temperatura acima da média: 8
Percentual da amostra: 0.41%


,Consumo_Eletrodomesticos,Consumo_Luzes,Temp_Cozinha,Umid_Cozinha,Temp_Sala,Umid_Sala,Temp_AreaServico,Umid_AreaServico
352,620,0,24.3900,44.3333,25.3700,37.7360,26.73,38.8633
616,570,10,24.8900,32.6667,23.5600,31.1400,23.60,30.8233
916,750,30,22.4300,43.0000,21.9267,40.1333,21.23,42.5967
993,690,10,22.5333,45.0267,22.0667,39.7900,22.39,41.9300
1182,600,20,24.8900,48.1333,28.9175,37.9475,27.10,42.4400


## 7. Comparação entre os dois DataFrames

Tabela comparativa com quantidade, percentual da amostra e médias de consumo e temperatura em cada conjunto.

In [12]:
comparativo = pd.DataFrame({
    "Critério": ["Só consumo > 70% do máximo", "Consumo > 70% E temperatura > média"],
    "Registros": [qtd_alto_consumo, qtd_duplo_criterio],
    "% da amostra": [round(percentual_alto_consumo, 2), round(percentual_duplo_criterio, 2)],
    "Consumo médio (Wh)": [
        round(df_alto_consumo["Consumo_Eletrodomesticos"].mean(), 2),
        round(df_duplo_criterio["Consumo_Eletrodomesticos"].mean(), 2),
    ],
    "Temp. média (°C)": [
        round(df_alto_consumo["Temp_Cozinha"].mean(), 2),
        round(df_duplo_criterio["Temp_Cozinha"].mean(), 2),
    ],
})

comparativo

,Critério,Registros,% da amostra,Consumo médio (Wh),Temp. média (°C)
0,Só consumo > 70% do máximo,22,1.11,630.00,21.67
1,Consumo > 70% E temperatura > média,8,0.41,643.75,23.46


In [13]:
registros_removidos = qtd_alto_consumo - qtd_duplo_criterio
reducao = (registros_removidos / qtd_alto_consumo) * 100
retidos = (qtd_duplo_criterio / qtd_alto_consumo) * 100

print("Registros eliminados pelo segundo critério:", registros_removidos)
print(f"Redução em relação ao primeiro DataFrame: {reducao:.2f}%")
print(f"Registros retidos: {retidos:.2f}%")

Registros eliminados pelo segundo critério: 14
Redução em relação ao primeiro DataFrame: 63.64%
Registros retidos: 36.36%


### Resposta interpretativa

O primeiro DataFrame usa um critério único e responde à pergunta "quando o consumo foi alto?". O segundo acrescenta a condição de temperatura e responde a uma pergunta mais restrita: "quando o consumo foi alto **e** o ambiente estava mais quente que o normal?".

Efeitos observados na amostra:

- **Redução do conjunto.** Como as duas condições precisam ser verdadeiras simultaneamente, o filtro combinado só pode manter registros que já estavam no primeiro conjunto — nunca acrescenta novos. O resultado é um subconjunto bem menor, com pouco mais de um terço dos registros originais.
- **Percentual muito baixo da amostra.** O recorte passa de pouco mais de 1% para menos de 0,5% do total, ou seja, a combinação descreve uma situação rara.
- **Perfil diferente dos registros.** A temperatura média do subconjunto sobe de forma clara, porque a condição elimina justamente os picos de consumo que ocorreram em ambiente frio ou na média. Já o consumo médio muda pouco: os dois grupos têm consumo alto por construção, então a temperatura não é o que separa consumos maiores de menores dentro dessa faixa.
- **Interpretação prática.** Existem picos de consumo tanto em temperatura alta quanto baixa. Isso indica que a temperatura sozinha não explica o consumo elevado — ela é um filtro adicional de contexto, não a causa. Como o conjunto final ficou muito pequeno, qualquer conclusão tirada dele tem baixa representatividade estatística e deve ser tratada com cautela.